# Phase 2: Supervised Fine-Tuning (SFT) - Exam & Interview Specialization

This notebook trains our model on Phase 2 data (Layer 4: GATE preparation, LeetCode-style DSA coding, ML theory interviews, and university viva Q&A) starting from the Phase 1 merged model. We use a lower rank adapter and lower learning rate to preserve the core foundations learned in Phase 1.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install Unsloth and training dependencies
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes xformers
!pip install datasets wandb

In [ ]:
# Setup logging and hubs
import wandb
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
except Exception:
    pass

try:
    wandb_key = userdata.get('WANDB_API_KEY')
    wandb.login(key=wandb_key)
except Exception:
    pass

In [ ]:
# Load Model and Tokenizer from Phase 1 Merged Weights
from unsloth import FastLanguageModel
import torch

max_seq_length = 4096
dtype = torch.float16
load_in_4bit = True

# Load the merged weights from Google Drive
model_path = "/content/drive/MyDrive/btech-ai-tutor/models/phase1_merged"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
# Configure LoRA Adapters (with reduced rank for specialized tuning)
model = FastLanguageModel.get_peft_model(
    model,
    r = 64,                # Lower rank to prevent catastrophic forgetting
    lora_alpha = 128,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

In [ ]:
# Load exam & interview training datasets from Drive
from datasets import load_dataset

dataset = load_dataset(
    "json", 
    data_files="/content/drive/MyDrive/btech-ai-tutor/data/phase2_train.jsonl", 
    split="train"
)
print(f"Loaded Phase 2 dataset with {len(dataset)} samples.")

In [ ]:
# Setup SFTTrainer
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 8,  # Effective batch size = 32
    warmup_ratio = 0.03,
    num_train_epochs = 3,             # Train for 3 epochs on the smaller dataset
    learning_rate = 5e-5,             # Reduced LR to preserve Phase 1 weights
    fp16 = True,
    bf16 = False,
    logging_steps = 10,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "cosine",
    seed = 42,
    output_dir = "/content/drive/MyDrive/btech-ai-tutor/checkpoints/phase2",
    save_strategy = "steps",
    save_steps = 200,
    save_total_limit = 2,
    report_to = "wandb" if wandb.run is not None else "none",
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = True,
    args = training_args,
)

In [ ]:
# Start training
trainer.train(resume_from_checkpoint=False)

In [ ]:
# Save LoRA adapter
adapter_path = "/content/drive/MyDrive/btech-ai-tutor/adapters/phase2_adapter"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"Saved adapter to {adapter_path}")

In [ ]:
# Merge and Save full model
merged_path = "/content/drive/MyDrive/btech-ai-tutor/models/phase2_merged"
model.save_pretrained_merged(merged_path, tokenizer, save_method = "merged_16bit")
print(f"Saved merged model to {merged_path}")